# Gold Layer 

### Import Libraries

In [7]:
# Import Libraries

from pyspark.sql.functions import monotonically_increasing_id
from pyspark.sql.functions import col

StatementMeta(, 351e3255-5b5e-4fb7-b1a0-c923031d1b07, 9, Finished, Available, Finished, False)

### DIM_PRODUCT

In [11]:
# DIM_PRODUCT 

## Lesen
df_product = spark.read.table("Silver_Lakehouse.dbo.Silver_Product")


## Transforamtion
dim_product = (
    df_product
    .select(
        "ProductKey",
        "Product",
        "Color",
        "Subcategory",
        "Category",
        "Standard_Cost",
        "load_timestamp",
        "source_system"
    )
    .dropDuplicates(["ProductKey"])
    .withColumn("ProductID", monotonically_increasing_id() + 1)
)
## Surrogate Key nach vorne
dim_product = dim_product.select(
    "ProductID",
    "ProductKey",
    "Product",
    "Color",
    "Subcategory",
    "Category",
    "Standard_Cost",
    "load_timestamp",
    "source_system"
)
## display
display(dim_product.limit(10))

StatementMeta(, 351e3255-5b5e-4fb7-b1a0-c923031d1b07, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 49f4164c-c340-4542-baeb-e7c870a98bd1)

In [15]:
##  DIM_PRODUCT In Gold Layer speichern
dim_product.write.mode("overwrite").saveAsTable("Gold_Lakehouse.dbo.Dim_Product")

StatementMeta(, 2116ba35-08d6-45ba-8a92-4dab576039f6, 17, Finished, Available, Finished, False)

### DIM_REGION

In [20]:
# DIM_REGION

## Lesen
df_region = spark.read.table("Silver_Lakehouse.dbo.Silver_Region")

dim_region = (
    df_region
    .select(
        "SalesTerritoryKey",
        "Region",
        "Country",
        "Group",
        "load_timestamp",
        "source_system"
    )
    .dropDuplicates()
    .withColumn("RegionID", monotonically_increasing_id() + 1)
)

## Surrogate Key nach vorne
dim_region = dim_region.select(
    "RegionID",
    "SalesTerritoryKey",
    "Region",
    "Country",
    "Group",
    "load_timestamp",
    "source_system"
)

display(
    spark.read.table("Gold_Lakehouse.dbo.Dim_Region").limit(10)
)

StatementMeta(, 351e3255-5b5e-4fb7-b1a0-c923031d1b07, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 224c08ac-9cdd-4642-b6dd-93df7f5e0c19)

In [18]:
## DIM_REGION in Gold Layer speichern
dim_region.write.mode("overwrite").saveAsTable("Gold_Lakehouse.dbo.Dim_Region")

StatementMeta(, 351e3255-5b5e-4fb7-b1a0-c923031d1b07, 20, Finished, Available, Finished, False)

### DIM_RESELLER

In [28]:
# DIM_RESELLER

## Reserller Lesen
df_reseller = spark.read.table("Silver_Lakehouse.dbo.Silver_Reseller")

dim_reseller = (
    df_reseller
    .select(
        "ResellerKey",
        "Reseller",
        "Business_Type",
        "City",
        "State_Province",
        "Country_Region",
        "load_timestamp",
        "record_source"
    )
    .dropDuplicates()
    .withColumn("ResellerID", monotonically_increasing_id() + 1)
)

## Surrogate Key nach vorne
dim_reseller = dim_reseller.select(
    "ResellerID",
    "ResellerKey",
    "Reseller",
    "Business_Type",
    "City",
    "State_Province",
    "Country_Region",
    "load_timestamp",
    "record_source"
)

display(dim_reseller.limit(10))

StatementMeta(, 2116ba35-08d6-45ba-8a92-4dab576039f6, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ee2cc21c-c8c0-4e3f-a43d-9e5847435793)

In [29]:
## DIM_RESELLER in Gold Layer speichern
dim_reseller.write.mode("overwrite").saveAsTable("Gold_Lakehouse.dbo.Dim_Reseller")

StatementMeta(, 2116ba35-08d6-45ba-8a92-4dab576039f6, 31, Finished, Available, Finished, False)

### DIM_SALESPERSON

In [31]:
# GalesPerson Dimension erstellen
## SalesPerson lesen
df_salesperson = spark.read.table("Silver_Lakehouse.dbo.Silver_Salesperson")

dim_salesperson = (
    df_salesperson
    .select(
        "EmployeeKey",
        "EmployeeID",
        "Salesperson",
        "Title",
        "UPN",
        "load_timestamp",
        "record_source"
    )
    .dropDuplicates()
    .withColumn("SalespersonID", monotonically_increasing_id() + 1)
)

## Surrogate Key nach vorne
dim_salesperson = dim_salesperson.select(
    "SalespersonID",
    "EmployeeKey",
    "EmployeeID",
    "Salesperson",
    "Title",
    "UPN",
    "load_timestamp",
    "record_source"
)

display(dim_salesperson.limit(10))

StatementMeta(, 2116ba35-08d6-45ba-8a92-4dab576039f6, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2dc8712a-585b-4325-98dc-c867851f213a)

In [32]:
## DIM_SalesPerson in Gold Layer speichern
dim_salesperson.write.mode("overwrite").saveAsTable("Gold_Lakehouse.dbo.Dim_Salesperson")

StatementMeta(, 2116ba35-08d6-45ba-8a92-4dab576039f6, 34, Finished, Available, Finished, False)

### FACT_SALES

In [26]:
# FACT_SALES 

## Lesen
df_sales = spark.read.table("Silver_Lakehouse.dbo.Silver_Sales")

dim_product = spark.read.table("Gold_Lakehouse.dbo.Dim_Product")
dim_region = spark.read.table("Gold_Lakehouse.dbo.Dim_Region")
dim_reseller = spark.read.table("Gold_Lakehouse.dbo.Dim_Reseller")
dim_salesperson = spark.read.table("Gold_Lakehouse.dbo.Dim_Salesperson")


## fact table befüllen
fact_sales = (
    df_sales
    .join(dim_product.select("ProductKey", "ProductID"), "ProductKey", "left")
    .join(dim_reseller.select("ResellerKey", "ResellerID"), "ResellerKey", "left")
    .join(dim_salesperson.select("EmployeeKey", "SalespersonID"), "EmployeeKey", "left")
    .join(dim_region.select("SalesTerritoryKey", "RegionID"), "SalesTerritoryKey", "left")
    .select(
        "SalesOrderNumber",
        "OrderDate",
        "ProductID",
        "ResellerID",
        "SalespersonID",
        "RegionID",
        "Quantity",
        "Unit_Price",
        "Sales",
        "Cost",
        "load_timestamp",
        "record_source"
    )
)

display(fact_sales.limit(10))

StatementMeta(, 351e3255-5b5e-4fb7-b1a0-c923031d1b07, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 45c1f9d3-221c-4e00-96c7-4da4bf097fbc)

In [27]:
## fact Sales in Gold Lazer speichern
fact_sales.write.mode("overwrite").saveAsTable(
    "Gold_Lakehouse.dbo.Fact_Sales"
)

StatementMeta(, 351e3255-5b5e-4fb7-b1a0-c923031d1b07, 29, Finished, Available, Finished, False)

### DIM_DATE

In [1]:
from pyspark.sql.functions import (
    col,
    year,
    quarter,
    month,
    dayofmonth,
    date_format,
    monotonically_increasing_id
)

df_sales = spark.read.table("Silver_Lakehouse.dbo.Silver_Sales")

dim_date = (
    df_sales
    .select("OrderDate")
    .dropDuplicates()
    .withColumn("DateID", monotonically_increasing_id() + 1)
    .withColumn("Year", year(col("OrderDate")))
    .withColumn("Quarter", quarter(col("OrderDate")))
    .withColumn("Month", month(col("OrderDate")))
    .withColumn("MonthName", date_format(col("OrderDate"), "MMMM"))
    .withColumn("Day", dayofmonth(col("OrderDate")))
)

dim_date = dim_date.select(
    "DateID",
    "OrderDate",
    "Year",
    "Quarter",
    "Month",
    "MonthName",
    "Day"
)

display(dim_date.limit(10))

StatementMeta(, 709964e5-b309-487d-b5d0-03b241ce9a64, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4e89b9ce-7a31-41ca-a628-fc153ef0c5e1)

In [3]:
dim_date.write.mode("overwrite").saveAsTable("Gold_Lakehouse.dbo.DIM_DATE")

StatementMeta(, 709964e5-b309-487d-b5d0-03b241ce9a64, 5, Finished, Available, Finished, False)